# Módulo 1 · Etapa 4 — Split por lesión
**TFM · HAM10000** · Responsable: **Adrián Arias** · MLflow: experimento `m1-datos`, run `m1-split-v1`, tag `fase=split`

**Objetivo:** tres conjuntos (entrenamiento, validación y prueba) **sin ninguna lesión compartida** y con proporciones de clase similares. Es la decisión más delicada del módulo y la que desbloquea al Módulo 2.

**Método** (el de `configs/base.yaml` y `docs/modulo1_datos.md`):
- `StratifiedGroupKFold` de scikit-learn: estratifica por clase (`dx`) y agrupa por lesión (`lesion_id`).
- 20 partes de ~5 %: 3 → prueba, 3 → validación, 14 → entrenamiento = **70 / 15 / 15**.
- Semilla `split_seed = 42` (de `base.yaml`).
- Todas las fotos de una lesión van al mismo conjunto. Se **mantienen todas** las fotos repetidas en entrenamiento (actúan como aumento natural).

**Por qué por lesión:** en la Etapa 2 vimos que el 79 % de las imágenes de melanoma pertenece a lesiones fotografiadas más de una vez. Un split por imagen pondría fotos del mismo melanoma en entrenamiento y en prueba, e inflaría la sensibilidad en melanoma.

**Reglas de esta etapa**
1. El split se genera **una sola vez**. Si ya existe en Drive, el notebook lo compara y **se niega a sobrescribirlo**.
2. Nadie lo regenera por su cuenta: aunque use la misma semilla, otra versión de scikit-learn podría producir otro reparto. Todos usan `split_v1.csv` y comprueban su huella `split_sha256`.
3. **No se mira el conjunto de prueba** más allá de contar cuántas imágenes tiene por clase.
4. **scikit-learn fijado en 1.9.1.** La versión 1.6.1 que trae Colab estratificaba mal (en la primera ejecución, `df` quedó con 11 imágenes en prueba). Si la desviación de composición supera **0,5 puntos**, el notebook se detiene sin guardar.

> **Historial:** una primera versión del split (huella `00ee7ee6…`, scikit-learn 1.6.1) se descartó antes de usarse por esa mala estratificación. Este notebook la aparta a `TFM/data/splits/descartado/` y marca su run como `descartado`; nunca la borra.

Entorno: **CPU**. Secretos: `DAGSHUB_USER`, `DAGSHUB_TOKEN`.

## 1. Preparación de la sesión

In [ ]:
AUTOR = "adrian"
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
DATASET_SHA256_ESPERADO = "06aee5e603d313ba69de38263053c792a1dff030eb9c44a96ceb477ceefb99a5"   # huella registrada en la Etapa 0
SKLEARN_VERSION = "1.9.1"                 # versión fijada para el split
SPLIT_DESCARTADO_SHA = "00ee7ee6852119d52e90ab3651d3c3cb0f3cfd1a26bc1cf7c47e50c9afdf28d8"  # 1ª versión (sklearn 1.6.1)
MAX_DESVIO_PP = 0.5                       # desviación máxima de composición aceptada

In [ ]:
import os, json, time
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/TFM")

REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!pip install -q "scikit-learn=={SKLEARN_VERSION}"

In [ ]:
# Solo se necesitan los metadatos (no las imágenes): se leen directamente del zip de Drive.
import zipfile, io
import pandas as pd
from src.tracking import load_config, file_sha256
cfg = load_config("configs/base.yaml")

META_LOCAL = Path("/content/HAM10000_metadata.csv")
if not META_LOCAL.exists():
    with zipfile.ZipFile(cfg["paths"]["dataset_zip"]) as z:
        META_LOCAL.write_bytes(z.read("HAM10000_metadata.csv"))
sha = file_sha256(META_LOCAL)
assert sha == DATASET_SHA256_ESPERADO, "[!] El CSV de metadatos no es el mismo que en la Etapa 0."
print("[+] Mismo dataset que en la Etapa 0")

meta = pd.read_csv(META_LOCAL)
SEMILLA = cfg["data"]["split_seed"]
PROP = cfg["data"]["proportions"]
print("Semilla:", SEMILLA, "| Proporciones:", PROP)
assert (SEMILLA, PROP) == (42, {"train": 0.70, "val": 0.15, "test": 0.15}), "[!] base.yaml cambió"

# Rutas de salida: repo (lo que se versiona) y Drive (copia persistente)
SPLIT_REPO = Path(cfg["paths"]["split_csv"])                  # data/splits/split_v1.csv
SPLIT_DRIVE = DRIVE_ROOT / "data" / "splits" / "split_v1.csv"
OUT = DRIVE_ROOT / "results" / "m1" / "etapa4"
for p in [SPLIT_REPO.parent, SPLIT_DRIVE.parent, OUT]:
    p.mkdir(parents=True, exist_ok=True)

import sklearn
print("scikit-learn", sklearn.__version__)
assert sklearn.__version__ == SKLEARN_VERSION, (
    f"[!] Está cargada scikit-learn {sklearn.__version__}, no {SKLEARN_VERSION}. "
    "Ve a Entorno de ejecución → Reiniciar sesión y ejecuta el notebook de nuevo desde arriba.")
print("[+] Versión fijada correcta")

## 2. Funciones del split

In [ ]:
# --- Lógica del split ---
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

CLASES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
SUBCONJUNTOS = ["train", "val", "test"]

def hacer_split(meta, semilla=42, n_folds=20, folds_test=3, folds_val=3):
    """Split estratificado por clase y agrupado por lesion_id.

    Se divide el dataset en n_folds partes de ~5 % cada una, sin que ninguna lesión
    quede repartida entre dos partes. Las primeras folds_test van a prueba, las
    siguientes folds_val a validación y el resto a entrenamiento (3 + 3 + 14 = 15/15/70).
    """
    meta = meta.sort_values("image_id").reset_index(drop=True)   # orden fijo: no depende del CSV
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=semilla)
    fold = np.full(len(meta), -1)
    for k, (_, idx) in enumerate(sgkf.split(meta, meta["dx"], groups=meta["lesion_id"])):
        fold[idx] = k
    assert (fold >= 0).all()
    split = np.where(fold < folds_test, "test",
                     np.where(fold < folds_test + folds_val, "val", "train"))
    out = meta[["image_id", "lesion_id", "dx"]].copy()
    out["split"] = split
    return out

def lesiones_compartidas(split):
    """Número de lesiones que aparecen en más de un subconjunto (debe ser 0)."""
    return int((split.groupby("lesion_id")["split"].nunique() > 1).sum())

def tablas_proporciones(split):
    img = pd.crosstab(split.dx, split.split).reindex(index=CLASES, columns=SUBCONJUNTOS).fillna(0).astype(int)
    les = (split.drop_duplicates("lesion_id")
                .pipe(lambda d: pd.crosstab(d.dx, d.split))
                .reindex(index=CLASES, columns=SUBCONJUNTOS).fillna(0).astype(int))
    pct_fila = (100 * img.div(img.sum(axis=1), axis=0)).round(1)        # % de cada clase en cada subconjunto
    pct_col = (100 * img / img.sum(axis=0)).round(2)                     # composición de cada subconjunto
    global_pct = (100 * img.sum(axis=1) / img.values.sum()).round(2)
    desvio = pct_col.sub(global_pct, axis=0).abs()                       # diferencia frente al dataset completo
    return img, les, pct_fila, pct_col, global_pct, desvio

## 3. Generar el split y verificarlo

In [ ]:
t0 = time.time()
split = hacer_split(meta, semilla=SEMILLA)
print(f"Split generado en {time.time()-t0:.1f} s")

# Comprobación 1 — determinismo: mismo resultado aunque el CSV venga en otro orden
split_bis = hacer_split(meta.sample(frac=1, random_state=123), semilla=SEMILLA)
assert split.equals(split_bis), "[!] El split no es determinista"
print("[+] Determinista: el orden del CSV no cambia el reparto")

# Comprobación 2 — test de no fuga: ninguna lesión en dos conjuntos (debe dar 0)
n_comp = lesiones_compartidas(split)
assert n_comp == 0, f"[!] {n_comp} lesiones aparecen en más de un conjunto"
print("[+] Lesiones compartidas entre conjuntos: 0")

# Comprobación 3 — no se pierde ni se duplica ninguna imagen
assert len(split) == len(meta) == 10015 and split.image_id.is_unique
assert set(split.split) == {"train", "val", "test"}
print("[+] 10.015 imágenes, cada una en un único conjunto")

## 4. Proporciones por clase

In [ ]:
img, les, pct_fila, pct_col, global_pct, desvio = tablas_proporciones(split)

tot = img.sum(axis=0)
print("Imágenes por conjunto:", tot.to_dict(), "→", (100 * tot / tot.sum()).round(2).to_dict(), "%")
lt = les.sum(axis=0)
print("Lesiones por conjunto:", lt.to_dict(), "→", (100 * lt / lt.sum()).round(2).to_dict(), "%\n")

tabla = pd.concat({"imágenes": img, "lesiones": les, "% de la clase": pct_fila}, axis=1)
display(tabla)
print("\nDiferencia máxima de composición frente al dataset completo (puntos porcentuales):",
      desvio.max().round(2).to_dict())

print("\nImágenes de cada clase en PRUEBA:", img["test"].to_dict())
print("Clase más pequeña en prueba:", img["test"].idxmin(), "con", img["test"].min(), "imágenes →",
      f"un acierto más o menos mueve su sensibilidad unos {100 / img['test'].min():.1f} puntos")
assert (img > 0).all().all(), "[!] Alguna clase no tiene imágenes en algún conjunto"
print("[+] Las 7 clases tienen representación en los tres conjuntos")
DESVIO_MAX = float(desvio.values.max())
assert DESVIO_MAX <= MAX_DESVIO_PP, f"[!] Desviación de {DESVIO_MAX:.2f} pp > {MAX_DESVIO_PP}: la estratificación falló."
print(f"[+] Desviación máxima {DESVIO_MAX:.2f} pp (límite {MAX_DESVIO_PP})")

## 5. Guardar el split (único y protegido)

In [ ]:
# Si en Drive está la 1ª versión descartada (y SOLO esa), se aparta a descartado/ sin borrarla.
if SPLIT_DRIVE.exists() and file_sha256(SPLIT_DRIVE) == SPLIT_DESCARTADO_SHA:
    destino = SPLIT_DRIVE.parent / "descartado" / "split_v1_sklearn161_00ee7ee6.csv"
    destino.parent.mkdir(exist_ok=True)
    SPLIT_DRIVE.rename(destino)
    print("Versión descartada apartada en:", destino)

if SPLIT_DRIVE.exists():
    previo = pd.read_csv(SPLIT_DRIVE)
    igual = previo.sort_values("image_id").reset_index(drop=True).equals(split.sort_values("image_id").reset_index(drop=True))
    print("Ya existía un split en Drive. ¿Idéntico al de ahora?:", igual)
    assert igual, "[!] El split de Drive es distinto. NO se sobrescribe."
    split = previo
else:
    split.to_csv(SPLIT_DRIVE, index=False)
    print("[+] Guardado en Drive:", SPLIT_DRIVE)

split.to_csv(SPLIT_REPO, index=False)                      # misma copia en la carpeta del repositorio
SPLIT_SHA = file_sha256(SPLIT_DRIVE)
assert SPLIT_SHA == file_sha256(SPLIT_REPO)
print("split_sha256:", SPLIT_SHA)

reporte = {
    "split_version": cfg["data"]["split_version"], "split_sha256": SPLIT_SHA,
    "dataset_sha256": sha, "split_seed": SEMILLA, "metodo": "StratifiedGroupKFold",
    "n_folds": 20, "folds": {"test": 3, "val": 3, "train": 14}, "agrupado_por": "lesion_id",
    "sklearn": sklearn.__version__, "lesiones_compartidas": n_comp, "desvio_max_pp": round(DESVIO_MAX, 3),
    "reemplaza_a": {"split_sha256": SPLIT_DESCARTADO_SHA, "motivo": "estratificación desequilibrada con scikit-learn 1.6.1 (df: 11 imágenes en test)"},
    "imagenes": {s: {c: int(img.loc[c, s]) for c in CLASES} for s in SUBCONJUNTOS},
    "lesiones": {s: {c: int(les.loc[c, s]) for c in CLASES} for s in SUBCONJUNTOS},
}
(OUT / "split_report.json").write_text(json.dumps(reporte, indent=2, ensure_ascii=False))
tabla.to_csv(OUT / "proporciones_split.csv")
print("[+] split_report.json y proporciones_split.csv en", OUT)

## 6. Registro en MLflow

In [ ]:
import mlflow
from src.tracking import setup_mlflow, standard_tags
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
setup_mlflow(cfg)
mlflow.set_experiment(cfg["mlflow"]["experiments"]["m1"])     # m1-datos

# Marca como descartado el run de la 1ª versión (no se borra: queda como historial)
viejos = mlflow.search_runs(filter_string=f"tags.split_sha256 = '{SPLIT_DESCARTADO_SHA}'", output_format="list")
for r in viejos:
    mlflow.MlflowClient().set_tag(r.info.run_id, "estado", "descartado")
    mlflow.MlflowClient().set_tag(r.info.run_id, "motivo", "estratificación desequilibrada con scikit-learn 1.6.1")
    print("Run anterior marcado como descartado:", r.info.run_id)

# standard_tags lee data/splits/split_v1.csv y pone su huella en split_sha256
tags = standard_tags(cfg, modulo="m1", fase="split", autor=AUTOR, dataset_sha256=sha)
assert tags["split_sha256"] == SPLIT_SHA
with mlflow.start_run(run_name="m1-split-v1", tags=tags) as run:
    mlflow.log_params({"split_seed": SEMILLA, "n_folds": 20, "folds_test": 3, "folds_val": 3,
                       "prop_train": 0.70, "prop_val": 0.15, "prop_test": 0.15,
                       "agrupado_por": "lesion_id", "fotos_repetidas_train": "todas",
                       "sklearn": sklearn.__version__})
    mlflow.log_metrics({"n_train": int(tot["train"]), "n_val": int(tot["val"]), "n_test": int(tot["test"]),
                        "lesiones_compartidas": n_comp,
                        "desvio_max_pp": DESVIO_MAX})
    mlflow.set_tag("reemplaza_a", SPLIT_DESCARTADO_SHA)
    mlflow.log_artifact(str(SPLIT_DRIVE))
    mlflow.log_artifacts(str(OUT), artifact_path="etapa4")
print("[+] Run m1-split-v1 registrado:", run.info.run_id)